In [1]:
!pip install pdfplumber pandas --quiet

import pdfplumber
import pandas as pd
import re
from pathlib import Path


In [2]:
from pathlib import Path

print("Current working directory:", Path.cwd())


Current working directory: c:\Users\Krishnendu Manna\Documents\new steup\Traffic-clusturing\city-traffic-clustering\notebooks


In [3]:
from pathlib import Path

PDF_DIR = Path(r"C:\Users\Krishnendu Manna\Documents\new steup\Traffic-clusturing\city-traffic-clustering\data\raw\bengaluru_signals")  # <-- put your pdf folder here

pdf_files = sorted(PDF_DIR.glob("*.pdf"))

print("PDF_DIR:", PDF_DIR)
print("Number of PDFs found:", len(pdf_files))
for f in pdf_files:
    print(" -", f.name)


PDF_DIR: C:\Users\Krishnendu Manna\Documents\new steup\Traffic-clusturing\city-traffic-clustering\data\raw\bengaluru_signals
Number of PDFs found: 134
 - Bengaluru-BPT-Adugodi-TPS-Signal-Timing-data-Aishwarya_Jn.pdf
 - Bengaluru-BPT-Adugodi-TPS-Signal-Timing-data-BDA_JN_KORAMANGALA_WATER_TANK.pdf
 - Bengaluru-BPT-Adugodi-TPS-Signal-Timing-data-Jaipuria_JN.pdf
 - Bengaluru-BPT-Adugodi-TPS-Signal-Timing-data-KRUPANIDHI_COLLEGE_MAHARAJA_JN.pdf
 - Bengaluru-BPT-Adugodi-TPS-Signal-Timing-data-NGV_REAR_GATE_JN_SONY_WORLD_KORAMANGALA_80FT_RD.pdf
 - Bengaluru-BPT-Adugodi-TPS-Signal-Timing-data-SRINIVAGILU_JN_UCO_BANK_JN.pdf
 - Bengaluru-BPT-Adugodi-TPS-Signal-Timing-data-WIPRO_JN_DEVARA_BEESANAHALLI_JN.pdf
 - Bengaluru-BPT-Airport-TPS-Signal-Timing-data-ISRO_JN_KADUBEESANAHALLI_JN.PDF
 - Bengaluru-BPT-Airport-TPS-Signal-Timing-data-KUVEMPU_CIRCLE_JN_MANIPAL_HOSPITAL_JN.pdf
 - Bengaluru-BPT-Airport-TPS-Signal-Timing-data-MARATHALLI_BRIDGE_JN_YAMALUR_JN.pdf
 - Bengaluru-BPT-Ashok-Nagar-TPS-Signa

In [4]:
def extract_intersection_header(text):
    """Extract intersection name + control type (FIXED, VAC, SYNC) from header lines."""
    for line in text.splitlines():
        line_clean = line.strip()
        m = re.search(r"(.*?)[\s-]*\b(FIXED|VAC|SYNC)\b", line_clean, re.IGNORECASE)
        if m:
            return m.group(1).strip(), m.group(2).upper()
    return None, None


def parse_value(val):
    """
    Convert values like '90 (70)' into:
    weekday=90, sunday=70
    or single values -> weekday=value, sunday=None
    """
    val = val.strip()
    if "(" in val:
        m = re.match(r"(\d+)\s*\((\d+)\)", val)
        if m:
            return int(m.group(1)), int(m.group(2))
    elif val.isdigit():
        return int(val), None
    return None, None


def extract_timing_rows(text):
    rows = []
    for line in text.splitlines():
        line = " ".join(line.split())
        
        # Match time format: HH:MM HH:MM ...
        m = re.match(r"(\d{2}:\d{2})\s+(\d{2}:\d{2})\s+(.*)", line)
        if not m:
            continue
        
        start, end, numbers_raw = m.groups()
        numbers = numbers_raw.split()

        parsed = [parse_value(n) for n in numbers]

        rows.append({
            "start_time": start,
            "end_time": end,
            "raw_values": parsed  # store temporarily
        })
    return rows


In [5]:
records = []

for pdf_path in pdf_files:
    print(f"Processing: {pdf_path.name}")
    
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            if not text.strip():
                continue
            
            # Extract metadata
            intersection_name, control_mode = extract_intersection_header(text)

            rows = extract_timing_rows(text)
            
            for row in rows:
                values = row["raw_values"]
                
                # Determine number of phases + cycle time
                weekday_vals = [v[0] for v in values if v[0] is not None]
                sunday_vals = [v[1] for v in values if v[1] is not None]

                record = {
                    "file": pdf_path.name,
                    "page": page_num,
                    "intersection": intersection_name,
                    "control_mode": control_mode,
                    "start_time": row["start_time"],
                    "end_time": row["end_time"],
                }

                # Assign phases dynamically
                for i, (wd, sd) in enumerate(values):
                    record[f"phase_{i+1}_weekday"] = wd
                    record[f"phase_{i+1}_sunday"] = sd
                
                records.append(record)

len(records)


Processing: Bengaluru-BPT-Adugodi-TPS-Signal-Timing-data-Aishwarya_Jn.pdf
Processing: Bengaluru-BPT-Adugodi-TPS-Signal-Timing-data-BDA_JN_KORAMANGALA_WATER_TANK.pdf
Processing: Bengaluru-BPT-Adugodi-TPS-Signal-Timing-data-Jaipuria_JN.pdf
Processing: Bengaluru-BPT-Adugodi-TPS-Signal-Timing-data-KRUPANIDHI_COLLEGE_MAHARAJA_JN.pdf
Processing: Bengaluru-BPT-Adugodi-TPS-Signal-Timing-data-NGV_REAR_GATE_JN_SONY_WORLD_KORAMANGALA_80FT_RD.pdf
Processing: Bengaluru-BPT-Adugodi-TPS-Signal-Timing-data-SRINIVAGILU_JN_UCO_BANK_JN.pdf
Processing: Bengaluru-BPT-Adugodi-TPS-Signal-Timing-data-WIPRO_JN_DEVARA_BEESANAHALLI_JN.pdf
Processing: Bengaluru-BPT-Airport-TPS-Signal-Timing-data-ISRO_JN_KADUBEESANAHALLI_JN.PDF
Processing: Bengaluru-BPT-Airport-TPS-Signal-Timing-data-KUVEMPU_CIRCLE_JN_MANIPAL_HOSPITAL_JN.pdf
Processing: Bengaluru-BPT-Airport-TPS-Signal-Timing-data-MARATHALLI_BRIDGE_JN_YAMALUR_JN.pdf
Processing: Bengaluru-BPT-Ashok-Nagar-TPS-Signal-Timing-data-ANEPALYA_JN_ARTS_and_CRAFT_CIRCLE_JN.p

1145

In [6]:
df = pd.DataFrame(records)
df.head()


,file,page,intersection,control_mode,start_time,end_time,phase_1_weekday,phase_1_sunday,phase_2_weekday,phase_2_sunday,phase_3_weekday,phase_3_sunday,phase_4_weekday,phase_4_sunday,phase_5_weekday,phase_5_sunday,phase_6_weekday,phase_6_sunday,phase_7_weekday,phase_7_sunday
0,Bengaluru-BPT-Adugodi-TPS-Signal-Timing-data-A...,1,"ADUGODIJUNCTION,ADUGODIPOLICESTATION",VAC,07:00,08:00,40.0,None,20.0,NaN,30.0,NaN,10.0,NaN,100.0,NaN,NaN,NaN,NaN,NaN
1,Bengaluru-BPT-Adugodi-TPS-Signal-Timing-data-A...,1,"ADUGODIJUNCTION,ADUGODIPOLICESTATION",VAC,08:00,11:00,NaN,None,NaN,NaN,NaN,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Bengaluru-BPT-Adugodi-TPS-Signal-Timing-data-A...,1,"ADUGODIJUNCTION,ADUGODIPOLICESTATION",VAC,16:30,21:00,NaN,None,NaN,NaN,NaN,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Bengaluru-BPT-Adugodi-TPS-Signal-Timing-data-A...,1,"ADUGODIJUNCTION,ADUGODIPOLICESTATION",VAC,21:00,23:00,40.0,None,20.0,NaN,30.0,NaN,8.0,NaN,98.0,NaN,NaN,NaN,NaN,NaN
4,Bengaluru-BPT-Adugodi-TPS-Signal-Timing-data-A...,1,"ADUGODIJUNCTION,ADUGODIPOLICESTATION",VAC,07:00,08:00,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1145 entries, 0 to 1144
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   file             1145 non-null   object 
 1   page             1145 non-null   int64  
 2   intersection     1141 non-null   object 
 3   control_mode     1141 non-null   object 
 4   start_time       1145 non-null   object 
 5   end_time         1145 non-null   object 
 6   phase_1_weekday  1050 non-null   float64
 7   phase_1_sunday   0 non-null      object 
 8   phase_2_weekday  1047 non-null   float64
 9   phase_2_sunday   0 non-null      float64
 10  phase_3_weekday  1051 non-null   float64
 11  phase_3_sunday   0 non-null      float64
 12  phase_4_weekday  970 non-null    float64
 13  phase_4_sunday   0 non-null      float64
 14  phase_5_weekday  776 non-null    float64
 15  phase_5_sunday   0 non-null      float64
 16  phase_6_weekday  353 non-null    float64
 17  phase_6_sunday

In [8]:
from pathlib import Path
from datetime import datetime

# Base directory (your root project folder)
BASE_DIR = Path(r"C:\Users\Krishnendu Manna\Documents\new steup\Traffic-clusturing\city-traffic-clustering\data")

# Ensure the processed folder exists
PROCESSED_DIR = BASE_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Optional timestamp for versioning
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
FILENAME = f"Bangalore_Signal_Timing_{timestamp}.csv"

# Final save path
SAVE_PATH = PROCESSED_DIR / FILENAME

# Save the DataFrame
df.to_csv(SAVE_PATH, index=False)

print(f"File saved successfully → {SAVE_PATH}")


File saved successfully → C:\Users\Krishnendu Manna\Documents\new steup\Traffic-clusturing\city-traffic-clustering\data\processed\Bangalore_Signal_Timing_2025-11-30_19-13.csv


In [12]:
df.head


<bound method NDFrame.head of Empty DataFrame
Columns: []
Index: []>